# Prétraitement des Données Immobilières

Ce notebook prépare les données immobilières pour la prédiction des prix en régression.

In [1]:
# Importation des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import re
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print("Librairies importées avec succès!")

Librairies importées avec succès!


In [2]:
# Charger les données brutes
df = pd.read_csv("../data/real_estate_data.csv")
print(f"Shape des données brutes: {df.shape}")
print("\nAperçu des données:")
df.head()

Shape des données brutes: (5653, 12)

Aperçu des données:


,titre,prix,prix_num,categorie,ville,localisation,type,nombre_p,time_posted,date_posted,month_posted,year_posted
0,À louer – Bureaux neufs S+1 et S+2 à Monastir ...,650 DT,650,Location Appartements,Monastir,Monastir,location,S+1,2/4/26 12:37,02-04-26,2,2026
1,S+1 haut standing pour la saison universitaire,850 DT,850,Location Appartements,Monastir,Monastir,location,S+1,8/30/25 10:49,08-30-25,8,2025
2,à vendre s+3 haut standing directement au prom...,350000 DT,350000,Vente Appartements,Monastir,Bekalta,vente,S+3,7/30/25 12:45,07-30-25,7,2025
3,Appartement s+1 vue sur mer pour vacance d’été...,1540 DT,1540,Location de vacances,Monastir,Bekalta,location,S+1,6/17/25 9:41,06-17-25,6,2025
4,Studio mignon estivale,55 DT,55,Location Appartements,Monastir,Monastir,location,S+0,6/10/25 20:59,06-10-25,6,2025


In [3]:
# Informations sur les données
print("Informations sur les données:")
df.info()

print("\nStatistiques descriptives:")
df.describe()

Informations sur les données:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5653 entries, 0 to 5652
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   titre         5653 non-null   object
 1   prix          5653 non-null   object
 2   prix_num      5653 non-null   int64 
 3   categorie     5653 non-null   object
 4   ville         5653 non-null   object
 5   localisation  5653 non-null   object
 6   type          5653 non-null   object
 7   nombre_p      5653 non-null   object
 8   time_posted   5653 non-null   object
 9   date_posted   5653 non-null   object
 10  month_posted  5653 non-null   int64 
 11  year_posted   5653 non-null   int64 
dtypes: int64(3), object(9)
memory usage: 530.1+ KB

Statistiques descriptives:


,prix_num,month_posted,year_posted
count,5.653000e+03,5653.000000,5653.000000
mean,2.602782e+05,5.818857,2025.185919
std,3.889321e+06,3.604828,0.389076
min,0.000000e+00,1.000000,2025.000000
25%,1.350000e+03,2.000000,2025.000000
50%,3.500000e+03,5.000000,2025.000000
75%,2.800000e+05,9.000000,2025.000000
max,2.100000e+08,12.000000,2026.000000


In [ ]:
# Nettoyage du prix
def clean_price(price_text):
    if pd.isna(price_text):
        return np.nan
    
    # Extraire les nombres du texte
    price_str = str(price_text)
    
    # Trouver tous les nombres dans le texte
    numbers = re.findall(r'\d+', price_str)
    
    if numbers:
        # Prendre le dernier nombre (généralement le prix)
        return int(numbers[-1])
    else:
        return np.nan

# Appliquer le nettoyage
df['price_cleaned'] = df['prix'].apply(clean_price)

# Comparaison avant/après
print("Nettoyage du prix:")
print(df[['prix', 'prix_num', 'price_cleaned']].head(10))

In [ ]:
# Nettoyage du nombre de pièces
def clean_rooms(nombre_p):
    if pd.isna(nombre_p):
        return np.nan
    
    rooms_str = str(nombre_p)
    
    # Chercher des patterns comme S+1, S+2, S+3, etc.
    s_pattern = re.search(r'S\+(\d+)', rooms_str)
    if s_pattern:
        return int(s_pattern.group(1)) + 1  # S+1 = 2 pièces
    
    # Chercher des nombres dans le texte
    numbers = re.findall(r'\d+', rooms_str)
    if numbers:
        # Prendre le premier nombre raisonnable (1-10)
        for num in numbers:
            if 1 <= int(num) <= 10:
                return int(num)
    
    return np.nan

# Appliquer le nettoyage
df['rooms_cleaned'] = df['nombre_p'].apply(clean_rooms)

print("Nettoyage du nombre de pièces:")
print(df[['titre', 'nombre_p', 'rooms_cleaned']].head(10))

In [ ]:
# Extraction du type de transaction
def extract_transaction_type(titre, type_col):
    if pd.isna(titre):
        return 'unknown'
    
    titre_str = str(titre).lower()
    type_str = str(type_col).lower()
    
    if 'louer' in titre_str or 'location' in type_str:
        return 'rent'
    elif 'vendre' in titre_str or 'vente' in type_str:
        return 'sale'
    else:
        return 'unknown'

# Appliquer l'extraction
df['transaction_type'] = df.apply(lambda row: extract_transaction_type(row['titre'], row['type']), axis=1)

print("Types de transaction:")
print(df['transaction_type'].value_counts())
print("\nTypes originaux:")
print(df['type'].value_counts())

In [ ]:
# Encodage des variables catégorielles
le_category = LabelEncoder()
le_city = LabelEncoder()
le_location = LabelEncoder()
le_transaction = LabelEncoder()

# Encoder les variables catégorielles
df['category_encoded'] = le_category.fit_transform(df['categorie'].fillna('unknown'))
df['city_encoded'] = le_city.fit_transform(df['ville'].fillna('unknown'))
df['location_encoded'] = le_location.fit_transform(df['localisation'].fillna('unknown'))
df['transaction_encoded'] = le_transaction.fit_transform(df['transaction_type'])

print("Encodage des variables catégorielles:")
print(f"Categories: {list(le_category.classes_)}")
print(f"Villes: {list(le_city.classes_)}")
print(f"Types de transaction: {list(le_transaction.classes_)}")

print("\nDistribution des catégories encodées:")
print(df['category_encoded'].value_counts())

In [ ]:
# Préparation des données finales
# Sélectionner les colonnes pertinentes
features = [
    'category_encoded',
    'city_encoded', 
    'location_encoded',
    'transaction_encoded',
    'rooms_cleaned',
    'month_posted',
    'year_posted'
]

# Créer le dataframe final
df_final = df[features + ['prix_num']].copy()

# Renommer les colonnes pour plus de clarté
df_final.columns = [
    'category',
    'city',
    'location',
    'type_transaction',
    'rooms',
    'post_month',
    'post_year',
    'price'
]

print("Dataset final:")
print(df_final.head())
print(f"\nShape final: {df_final.shape}")

# Vérifier les types
print("\nTypes de données finaux:")
print(df_final.dtypes)

In [ ]:
# Gestion des valeurs manquantes
print("Valeurs manquantes avant nettoyage:")
print(df_final.isnull().sum())

# Remplir les valeurs manquantes
df_final['rooms'] = df_final['rooms'].fillna(df_final['rooms'].median())
df_final = df_final.dropna()  # Supprimer les lignes avec prix manquant

print("\nValeurs manquantes après nettoyage:")
print(df_final.isnull().sum())
print(f"\nShape après nettoyage: {df_final.shape}")

# Statistiques descriptives finales
print("\nStatistiques descriptives finales:")
print(df_final.describe())

In [ ]:
# Visualisation des données
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribution des prix
axes[0, 0].hist(df_final['price'], bins=50, alpha=0.7)
axes[0, 0].set_title('Distribution des Prix')
axes[0, 0].set_xlabel('Prix')
axes[0, 0].set_ylabel('Fréquence')

# Distribution des pièces
axes[0, 1].hist(df_final['rooms'], bins=20, alpha=0.7)
axes[0, 1].set_title('Distribution des Pièces')
axes[0, 1].set_xlabel('Nombre de Pièces')
axes[0, 1].set_ylabel('Fréquence')

# Prix par type de transaction
transaction_prices = df_final.groupby('type_transaction')['price'].mean()
axes[1, 0].bar(transaction_prices.index, transaction_prices.values)
axes[1, 0].set_title('Prix Moyen par Type de Transaction')
axes[1, 0].set_xlabel('Type de Transaction')
axes[1, 0].set_ylabel('Prix Moyen')

# Prix par ville (top 10)
city_prices = df_final.groupby('city')['price'].mean().sort_values(ascending=False)[:10]
axes[1, 1].bar(range(len(city_prices)), city_prices.values)
axes[1, 1].set_title('Top 10 des Prix par Ville')
axes[1, 1].set_xlabel('Ville')
axes[1, 1].set_ylabel('Prix Moyen')

plt.tight_layout()
plt.show()

In [ ]:
# Normalisation des données (optionnel)
scaler = StandardScaler()

# Séparer features et target
X = df_final.drop('price', axis=1)
y = df_final['price']

# Normaliser les features
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("Données normalisées:")
print(X_scaled_df.head())
print("\nStatistiques après normalisation:")
print(X_scaled_df.describe())

In [ ]:
# Sauvegarder les données prétraitées
# Version sans normalisation (pour les modèles)
df_final.to_csv('../data/real_estate_processed_clean.csv', index=False)

# Version normalisée
df_scaled = X_scaled_df.copy()
df_scaled['price'] = y
df_scaled.to_csv('../data/real_estate_processed_scaled.csv', index=False)

# Sauvegarder les encodeurs et scaler
import joblib
import os

if not os.path.exists('../models'):
    os.makedirs('../models')

joblib.dump(le_category, '../models/le_category.pkl')
joblib.dump(le_city, '../models/le_city.pkl')
joblib.dump(le_location, '../models/le_location.pkl')
joblib.dump(le_transaction, '../models/le_transaction.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("Données prétraitées sauvegardées avec succès!")
print("- real_estate_processed_clean.csv (sans normalisation)")
print("- real_estate_processed_scaled.csv (avec normalisation)")
print("- Encodeurs et scaler sauvegardés dans ../models/")

In [ ]:
# Résumé final
print("=== RÉSUMÉ DU PRÉTRAITEMENT ===")
print(f"Dataset initial: {df.shape}")
print(f"Dataset final: {df_final.shape}")
print(f"Taux de conservation: {len(df_final)/len(df)*100:.1f}%")
print()
print("Features finales:")
for col in X.columns:
    print(f"  - {col}: {X[col].dtype}")
print()
print("Target:")
print(f"  - price: {y.dtype}")
print()
print("Statistiques du prix:")
print(f"  - Minimum: {y.min():.0f}")
print(f"  - Maximum: {y.max():.0f}")
print(f"  - Moyenne: {y.mean():.0f}")
print(f"  - Médiane: {y.median():.0f}")
print()
print("PRÉTRAITEMENT TERMINÉ AVEC SUCCÈS! ")
print("\nLes notebooks de ML peuvent maintenant utiliser real_estate_processed_clean.csv")

In [ ]:
# Sauvegarder les données prétraitées
# Version sans normalisation (pour les modèles)
df_final.to_csv('../data/real_estate_processed_clean.csv', index=False)

# Version normalisée
df_scaled = X_scaled_df.copy()
df_scaled['price'] = y
df_scaled.to_csv('../data/real_estate_processed_scaled.csv', index=False)

# Sauvegarder les encodeurs et scaler
import joblib
import os

if not os.path.exists('../models'):
    os.makedirs('../models')

joblib.dump(le_category, '../models/le_category.pkl')
joblib.dump(le_city, '../models/le_city.pkl')
joblib.dump(le_location, '../models/le_location.pkl')
joblib.dump(le_transaction, '../models/le_transaction.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("Données prétraitées sauvegardées avec succès!")
print("- real_estate_processed_clean.csv (sans normalisation)")
print("- real_estate_processed_scaled.csv (avec normalisation)")
print("- Encodeurs et scaler sauvegardés dans ../models/")

In [ ]:
# Résumé final
print("=== RÉSUMÉ DU PRÉTRAITEMENT ===")
print(f"Dataset initial: {df.shape}")
print(f"Dataset final: {df_final.shape}")
print(f"Taux de conservation: {len(df_final)/len(df)*100:.1f}%")
print()
print("Features finales:")
for col in X.columns:
    print(f"  - {col}: {X[col].dtype}")
print()
print("Target:")
print(f"  - price: {y.dtype}")
print()
print("Statistiques du prix:")
print(f"  - Minimum: {y.min():.0f}")
print(f"  - Maximum: {y.max():.0f}")
print(f"  - Moyenne: {y.mean():.0f}")
print(f"  - Médiane: {y.median():.0f}")
print()
print("PRÉTRAITEMENT TERMINÉ AVEC SUCCÈS! ")